- 지역별 재난 특성 비교 (polar)
- 시도별 재난 종류 분포 (area)
- 재난 종류별 월별 특보 발생 패턴 (hitmap)
- 재난 종류별 특보 발생 지역 (scatter)

In [4]:
import pandas as pd
df = pd.read_csv(r"C:\project_dashboard\preprocessing_data\preprocessing\danger_clean.csv", encoding="utf-8-sig")
df.head()

,발표시간,지역,시군구,재난종류,특보등급,해당지역
0,2023-01-01 10:00,경북,군위,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
1,2023-01-01 10:00,경북,안동,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
2,2023-01-01 10:00,경북,영주,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
3,2023-01-01 10:00,경북,의성,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...
4,2023-01-01 10:00,경북,청송,한파,주의보,(1) 한파주의보 발표 : 경상북도(군위. 안동. 영주. 의성. 청송. 영양평지. ...


In [5]:
# 지역 + 특보등급 + 재난종류 발생 건수
result = (
    df.groupby(["지역", "특보등급", "재난종류"])
      .size()
      .reset_index(name="발생건수")
      .sort_values(["지역", "특보등급", "발생건수"], ascending=[True, True, False])   #
)


In [6]:
import plotly.express as px
import plotly.graph_objects as go

radar = (
    result.groupby(["지역", "재난종류"])["발생건수"]
    .sum()
    .unstack()
    .fillna(0)
)

colors = {
    "경남": "#1f77b4",
    "경북": "#ff7f0e",
    "대구": "#2ca02c",
    "부산": "#d62728",
    "울산": "#9467bd"
}

fig = go.Figure()

for region in radar.index:
    fig.add_trace(go.Scatterpolar(
        r=radar.loc[region],
        theta=radar.columns,
        fill="toself",
        name=region,
        line=dict(color=colors.get(region, "#333333"))
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title="지역별 재난 특성 비교"
)

fig.show()

In [7]:


df["발표시간"] = pd.to_datetime(df["발표시간"])
df["년월"] = df["발표시간"].dt.to_period("M").astype(str)

month_type = (
    df.groupby(["재난종류", "년월"])
      .size()
      .reset_index(name="발생건수")
)

heat = month_type.pivot(
    index="재난종류",
    columns="년월",
    values="발생건수"
).fillna(0)

# 10 이상만 숫자 표시
text_values = heat.map(lambda x: f"{int(x)}" if x >= 10 else "")

fig = px.imshow(
    heat,
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="재난종류별 월별 특보 발생 패턴"
)

fig.update_traces(
    text=text_values,
    texttemplate="%{text}",
    textfont={"size":11, "color":"black"},
    hovertemplate="재난종류: %{y}<br>년월: %{x}<br>발생건수: %{z}<extra></extra>"
)

fig.update_layout(
    width=1400,
    height=500,
    xaxis_title="년월",
    yaxis_title="재난종류",
    coloraxis_colorbar_title="발생건수",
    title_x=0.5
)

fig.show()

In [8]:

# 재난 종류별 특보가 많이 발생한 지역 건수
df_type = df[["재난종류","특보등급","시군구"]].dropna()

calamity_type = df_type.groupby(["재난종류", "시군구"]).size().reset_index(name="발생건수").sort_values(["재난종류", "발생건수"], ascending=[True, False])

calamity_type.groupby("재난종류").all()

top3 = (
    calamity_type
    .groupby(["재난종류", "시군구"], as_index=False)["발생건수"]
    .sum()
)

In [9]:
fig = px.scatter(
    top3,
    x="재난종류",
    y="시군구",
    size="발생건수",
    color="시군구",
    title="재난종류별 특보 발생 지역",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()